# Deviazione standard della proprietà target

Questo notebook calcola la deviazione standard della proprietà scelta nel dataset. Il valore viene usato come riferimento per capire quanto sono dispersi i dati e, quando serve, per normalizzare alcune metriche dei modelli.

La procedura parte dai documenti salvati in MongoDB, porta le proprietà in colonne normali, calcola i descrittori strutturali e aggiunge le fingerprint. Dopo la pulizia dei dati, separa il target dalle feature e calcola la deviazione standard dei valori disponibili.

Per cambiare proprietà basta aggiornare la variabile `target` nella cella di configurazione.

In [ ]:
from pathlib import Path
import pandas as pd
import warnings
import numpy as np
warnings.filterwarnings("ignore")

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()
while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent
if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

## Estrazione dati da MongoDB

In [ ]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(ROOT / ".env")
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["PolymerPrediction"]
collection = db["biceranoPolymers"]

#print(collection.count_documents({}))

In [ ]:
docs = list(collection.find({}))
df = pd.DataFrame(docs)

# Porto le proprieta annidate in colonne normali
prop_df = pd.json_normalize(df["properties"])

# Integro i dati nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

target="Tg"
# rimuovo righe senza SMILES o target
df = df.dropna(subset=["smiles", target])
df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()

df.head()

## Integrazione descrittori RDKit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import numpy as np

descriptor_names = [desc[0] for desc in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
# print(descriptor_names)

def rdkit_descr(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return {d: np.nan for d in descriptor_names}
    values = calc.CalcDescriptors(mol)
    return dict(zip(descriptor_names, values))

desc_df = df["smiles"].apply(rdkit_descr).apply(pd.Series)
desc_df.head()

## Aggiunta Fingerprint

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def morgan_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return [0]*1024
    fp = morgan_gen.GetFingerprint(mol)  
    return list(fp)

fp_df = df["smiles"].apply(morgan_fp).apply(pd.Series)
fp_df.columns = [f"fp_{i}" for i in range(fp_df.shape[1])]

In [ ]:
from rdkit.Chem import MACCSkeys

def maccs_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return [0]*167  # MACCS ha 167 bit (indice 0 inutilizzato)
    fp = MACCSkeys.GenMACCSKeys(mol)
    return list(fp)
maccs_df = df["smiles"].apply(maccs_fp).apply(pd.Series)
maccs_df.columns=[f"maccs_{i}" for i in range(167)]

feature_df = pd.concat([desc_df, fp_df, maccs_df], axis=1)

## Pulizia dei dati

In [ ]:
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))
feature_df = feature_df.dropna(axis=1, how="all")

# Rimuovo le colonne costanti 
from sklearn.feature_selection import VarianceThreshold

sel = VarianceThreshold(threshold=0.0)
sel.fit(feature_df)

mask = sel.get_support()
feature_df = feature_df.loc[:, mask]

print("Shape:", feature_df.shape[1])

## Preparazione dataset (separazione target)

In [ ]:
y = df[target]
X = feature_df.copy()
X.index = df["_id"]
print("Shape X:", X.shape)
print("Shape y:", y.shape)
X.sample(10)

import numpy as np

X_np = np.ascontiguousarray(X.to_numpy(), dtype=np.float32)

print("dtype:", X_np.dtype)
print("contiguous:", X_np.flags['C_CONTIGUOUS'])
print("finite:", np.isfinite(X_np).all())
print("max abs:", np.max(np.abs(X_np)))


## Calcolo deviazione standard

In [ ]:
std = y.std(ddof=1)
print(y.min(), y.max())
print(f"Deviazione standard {target}: {std:.4f}")
